In [ ]:
# library imports
import pandas as pd                             # tables, dataframes
import numpy as np                              # arrays, numerical computations
from uncertainties import ufloat as uf          # uncertainty propagation

from pathlib import Path                        # paths in operating system
# path config
downloads_dir = Path.home() / 'Downloads'
# make sure the folder exists (just in case)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'my_plot.pdf'

In [ ]:
# custom functions

# type A uncertainty
def unc_A(data):
    '''
    Type A uncertainty (standard error of the mean)

    data : list or numpy array
        measurements
    return : float
        type A uncertainty
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)

    return u_A

# loading from df into numpy, needs .T for the correct allocation
def df_to_numpy_matrix(df, column, dtype=float):
    '''
    Converts the selected columns of a dataframe to a numpy array

    df : dataframe
        dataframe to be converted to a numpy object
    column : str or list[str]
        column name (str) or list of column names (list[str])
    dtype : type
        data type of the resulting array (default float)
    return : numpy.ndarray
        matrix of the selected columns; 1D array if a single column is given
    '''
    if isinstance(column, str):
        column = [column]

    missing = [s for s in column if s not in df.columns]
    if missing:
        raise KeyError(f'Missing columns: {missing}')

    data = df[column].to_numpy(dtype=dtype)

    return data[:, 0] if len(column) == 1 else data

# loading from df into individual arrays
def df_to_arrays(df, column, first_row=0, last_row=None, dtype=float):
    '''
    Extracts the selected columns of a dataframe as separate numpy arrays.
    NaN values are dropped from each column.

    df : dataframe
        source dataframe
    column : str or list[str]
        column name (str) or list of column names (list[str])
    first_row : int
        first row (index) to include
    last_row : int or None
        row (index) at which to stop (exclusive); None = until the end
    dtype : type
        data type of the resulting arrays (default float)
    return : list[numpy.ndarray]
        one array per column
    '''
    if isinstance(column, str):
        column = [column]

    missing = [s for s in column if s not in df.columns]
    if missing:
        raise KeyError(f'Missing columns: {missing}')

    return [df[s].iloc[first_row:last_row].dropna().to_numpy(dtype=dtype) for s in column]

# ufloat from the arithmetic mean
def ufloat_unc_A(data):
    '''
    Converts a dataset to a ufloat (arithmetic mean +- type A uncertainty)

    data : list or numpy array
        dataset of measurements
    return : ufloat
    '''

    return uf(np.mean(data), unc_A(data))

# uarray to LaTeX
def uarray_to_latex(*array, digit=1, format='u', style='L', only_return=False):
    '''
    Formats ufloats as LaTeX strings (e.g. $1.23 \\pm 0.04$)

    *array : ufloat
        one or more ufloats to format
    digit : int
        number of digits / precision in the format specification
    format : str
        ufloat format type (e.g. 'u' for the uncertainty in parentheses)
    style : str
        output style; 'L' for LaTeX
    only_return : bool
        True  -> return the list of strings
        False -> print each string
    return : list[str] or None
        formatted strings if only_return=True, otherwise None
    '''
    output = [f'${u:.{digit}{format}{style}}$' for u in array]
    if only_return:
        return output
    else:
        for s in output:
            print(s)

In [ ]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_tenk = pd.read_csv(url)

In [ ]:
# tenka vrstva
vln_delka_tenk = 532.1*1e-9
df_tenk_list = ['x1a [mm]', 'x2a [mm]', 'x1b [mm]', 'x2b [mm]', 'x1c [mm]', 'x2c [mm]', 'x1d [mm]', 'x2d [mm]', 'x1e [mm]', 'x2e [mm]', 'x1f [mm]', 'x2f [mm]', 'x1g [mm]', 'x2g [mm]', 'x1h [mm]', 'x2h [mm]']
x1a, x2a, x1b, x2b, x1c, x2c, x1d, x2d, x1e, x2e, x1f, x2f, x1g, x2g, x1h, x2h = df_to_arrays(df_tenk, df_tenk_list, dtype=int)

for u in (x2h/x1h): print(f'{u:.3f}')

def tenka(pripona):
    x1 = globals()[f"x1{pripona}"]
    x2 = globals()[f"x2{pripona}"]
    return (x2 / x1) * (vln_delka_tenk / 2)

from string import ascii_lowercase

n=8

stred = np.concatenate([tenka('a'),tenka('b')])
print('Střed:', uarray_to_latex(ufloat_unc_A(stred)*1e9, only_return=True))

dole = np.concatenate([tenka('c'),tenka('d')])
print('Dole:', uarray_to_latex(ufloat_unc_A(dole)*1e9, only_return=True))

nahore = np.concatenate([tenka('e'),tenka('f')])
print('Nahoře:', uarray_to_latex(ufloat_unc_A(nahore)*1e9, only_return=True))

dalsi_stred = np.concatenate([tenka('g'),tenka('h')])
print('Další střed:', uarray_to_latex(ufloat_unc_A(dalsi_stred)*1e9, only_return=True))

for k in ascii_lowercase[:n]:
    print(k, ufloat_unc_A(tenka(k)))

vysledek = []
for k in ascii_lowercase[:n]:
    vysledek.extend(tenka(k))


tlouska = ufloat_unc_A(vysledek)
uarray_to_latex(tlouska*1e9, digit=1, format='u')

In [ ]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "1193928494"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_vzduch = pd.read_csv(url)

In [ ]:
# index lomu vzduchu
d = 0.04
p0 = 99600
kilopond_1 = uf(0.005, 0.010)
kilopond_2 = uf(1.020, 0.010)
kilopond_to_Pa = 98066.5
deltap = (kilopond_2 - kilopond_1) * kilopond_to_Pa
#N = np.array(df_vzduch['Počet posunutých proužků N'])
N = uf(38, 1)
n_vz = 1 + ((N * vln_delka_tenk)/(2*d))*(p0/deltap)
#n_vz = ufloat_unc_A(n_vz)
uarray_to_latex(n_vz)
print(deltap)
uarray_to_latex(deltap, digit=2)

n_ref_vz = 1.00026571
jedna = 1
print(((-(n_vz.n - jedna) + (n_ref_vz - jedna)) / (n_ref_vz - jedna))*100)

In [ ]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "1791918306"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_mrizka = pd.read_csv(url)

In [ ]:
# difrakce na mrizce
vln_delka_dif = 632.8*1e-9

df_mrizka_list = ['Vzdálenost mřížka–stínítko xa [mm]', 'y1a vpravo [mm]', 'y1a vlevo [mm]', 'y2a vpravo [mm]', 'y2a vlevo [mm]', 'Vzdálenost mřížka–stínítko xb [mm]', 'y1b vpravo [mm]', 'y1b vlevo [mm]', 'y2b vpravo [mm]', 'y2b vlevo [mm]']
xa, y1a_right, y1a_left, y2a_right, y2a_left, xb, y1b_right, y1b_left, y2b_right, y2b_left = df_to_arrays(df_mrizka, df_mrizka_list, dtype=int)
y1a = (y1a_left + y1a_right)/2
y2a = (y2a_left + y2a_right)/2
y1b = (y1b_left + y1b_right)/2
y2b = (y2b_left + y2b_right)/2

def difraction_formule(y, x, m):
    sin_alpha = y/((y**2 + x**2)**0.5)
    d = (m * vln_delka_dif)/sin_alpha
    return d

d_a = np.concatenate([difraction_formule(y1a, xa, 1), difraction_formule(y2a, xa, 2)])
d_a = uf(np.mean(d_a), unc_A(d_a))
uarray_to_latex(d_a, digit=1, format='u')
uarray_to_latex(1/d_a/1000, digit=1)
d_b = np.concatenate([difraction_formule(y1b, xb, 1), difraction_formule(y2b, xb, 2)])
d_b = uf(np.mean(d_b), unc_A(d_b))
uarray_to_latex(d_b, digit=1, format='u')
uarray_to_latex(1/d_b/1000, digit=1)

for u in difraction_formule(y1b, xb, 1): print(f'{1/u*1e-3:.1f}')
for u in difraction_formule(y2b, xb, 2): print(f'{1/u*1e-3:.1f}')